# 🤝 マルチエージェント（ディスカッション型） with Phi-3.5

3つのエージェントがお互いの出力を参照しながらタスクを処理します。

```
ユーザーのタスク
       ↓
┌──────────────────────────────────┐
│  🔍 調査エージェント              │
│  テーマに関する情報を収集する      │
└──────────────┬───────────────────┘
               ↓ 調査結果を渡す
┌──────────────────────────────────┐
│  📊 分析エージェント              │
│  調査結果を受け取り分析・評価する  │
│  → 調査エージェントに追加調査依頼 │  ← ディスカッション
└──────────────┬───────────────────┘
               ↓ 分析結果を渡す
┌──────────────────────────────────┐
│  ✍️  執筆エージェント             │
│  調査・分析を統合して最終回答作成  │
└──────────────────────────────────┘
       ↓
    最終回答
```

| 項目 | 内容 |
|------|------|
| **推論エンジン** | llama-cpp-python |
| **LLM** | Phi-3.5-mini-instruct |
| **エージェント数** | 3（調査・分析・執筆） |
| **連携方式** | ディスカッション型（相互参照） |

## Step 1: インストール

In [ ]:
import subprocess, os
subprocess.run(
    ['pip', 'install', 'llama-cpp-python',
     '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121',
     '--upgrade', '-q'],
    env={**os.environ, 'CMAKE_ARGS': '-DGGML_CUDA=on'},
    capture_output=True,
)
print('✅ インストール完了 → ランタイムを再起動してください')

⚠️ **ランタイムを再起動してから Step 2 以降を実行してください**

## Step 2: GPU確認 & モデルダウンロード

In [ ]:
import torch
from llama_cpp import Llama

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} ({vram:.1f} GB VRAM)')
else:
    print('⚠️  GPU未検出')
print('✅ llama-cpp-python OK')

In [ ]:
%%bash
MODEL_PATH="/content/phi-3.5-mini-q4.gguf"
MODEL_URL="https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf"
if [ -f "$MODEL_PATH" ]; then
  echo "✅ モデルは既にダウンロード済みです"
else
  echo "📥 ダウンロード中... (~2.2GB)"
  wget -q --show-progress -O "$MODEL_PATH" "$MODEL_URL"
  echo "✅ ダウンロード完了"
fi
ls -lh "$MODEL_PATH"

## Step 3: モデルロード

In [ ]:
from llama_cpp import Llama

print('📥 Phi-3.5 をロード中...')
llm = Llama(
    model_path='/content/phi-3.5-mini-q4.gguf',
    n_gpu_layers=35,
    n_ctx=4096,
    n_batch=512,
    verbose=False,
    chat_format='chatml',
)
print('✅ モデルロード完了')

resp = llm.create_chat_completion(
    messages=[{'role': 'user', 'content': '「元気です」を英語1語で。'}],
    max_tokens=10, temperature=0.1,
)
print(f'✅ 動作確認: {resp["choices"][0]["message"]["content"].strip()}')

## Step 4: 共通LLM呼び出し関数 & ツール定義

In [ ]:
import re, json, math

def llm_chat(messages: list, max_tokens: int = 512, temperature: float = 0.5) -> str:
    resp = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        repeat_penalty=1.1,
        stop=['<|end|>', '<|user|>'],
    )
    return resp['choices'][0]['message']['content'].strip()

# ツール（調査エージェントが使用）
def search_knowledge(query: str) -> str:
    kb = {
        'python':    'Pythonは1991年にGuido van Rossumが開発した高水準言語。読みやすい文法と豊富なライブラリが特徴。',
        '機械学習':  '機械学習はデータからパターンを自動的に学習するAI技術。教師あり・教師なし・強化学習の3種がある。',
        'phi':       'Phi-3.5はMicrosoftが開発した3.8Bパラメータの小型高性能LLM。MITライセンスで登録不要・商用利用可能。',
        'llama':     'llama-cpp-pythonはC++製LLM推論エンジンllama.cppのPythonバインディング。pip installだけで使えてGPU加速対応。',
        '深層学習':  '深層学習は多層ニューラルネットワークを用いる機械学習の一手法。画像認識・自然言語処理で革新的な性能を達成した。',
        '東京':      '東京は日本の首都。人口約1,400万人（都区部）。世界最大級の都市圏の一つ。',
    }
    for key, val in kb.items():
        if key.lower() in query.lower():
            return val
    return f'「{query}」に関する情報は見つかりませんでした。'

def calculator(expression: str) -> str:
    try:
        clean = re.sub(r'[^0-9\.\+\-\*\/\(\)\s]', '', expression)
        result = eval(clean, {'__builtins__': {}, 'math': math})
        return f'{expression} = {result}'
    except Exception as e:
        return f'計算エラー: {e}'

print('✅ 共通関数・ツール定義完了')

## Step 5: 3つのエージェントを定義

In [ ]:
# ============================================================
# 🔍 調査エージェント
# 役割: テーマに関する情報を収集し、分析エージェントからの
#       追加調査依頼にも応答する
# ============================================================

class ResearchAgent:
    def __init__(self):
        self.name = '🔍 調査エージェント'
        self.system_prompt = """あなたは調査専門のAIエージェントです。
与えられたテーマについて情報を収集・整理して報告します。
事実に基づいて簡潔に日本語で回答してください。"""

    def research(self, topic: str, additional_request: str = None) -> str:
        """テーマを調査する。additional_requestがあれば追加調査も行う"""
        # ナレッジベースから情報取得
        kb_result = search_knowledge(topic)

        user_content = f'テーマ: {topic}\n\nナレッジベースの情報: {kb_result}'
        if additional_request:
            user_content += f'\n\n分析エージェントからの追加調査依頼: {additional_request}'

        return llm_chat([
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user',   'content': user_content},
        ], max_tokens=300)


# ============================================================
# 📊 分析エージェント
# 役割: 調査結果を受け取り分析・評価する。
#       不足があれば調査エージェントに追加調査を依頼する
# ============================================================

class AnalysisAgent:
    def __init__(self):
        self.name = '📊 分析エージェント'
        self.system_prompt = """あなたは分析専門のAIエージェントです。
調査結果を批判的・多角的に分析し、重要なポイントと課題を整理します。
情報が不足している場合は、追加調査が必要な点を明示してください。
日本語で回答してください。"""

    def analyze(self, topic: str, research_result: str) -> dict:
        """調査結果を分析し、追加調査が必要かどうかも返す"""
        response = llm_chat([
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user', 'content':
                f'テーマ: {topic}\n\n'
                f'調査結果:\n{research_result}\n\n'
                f'上記を分析してください。情報が不足している場合は '
                f'「追加調査依頼:」に続けて依頼内容を書いてください。'},
        ], max_tokens=400)

        # 追加調査依頼が含まれているか抽出
        additional_request = None
        if '追加調査依頼:' in response:
            parts = response.split('追加調査依頼:')
            additional_request = parts[1].strip() if len(parts) > 1 else None
            response = parts[0].strip()

        return {
            'analysis': response,
            'additional_request': additional_request,
        }


# ============================================================
# ✍️  執筆エージェント
# 役割: 調査・分析の全結果を統合して最終回答を作成する
# ============================================================

class WritingAgent:
    def __init__(self):
        self.name = '✍️  執筆エージェント'
        self.system_prompt = """あなたは執筆専門のAIエージェントです。
調査・分析の結果を統合し、読みやすく分かりやすい最終回答を日本語で作成します。
構造化された回答を心がけてください。"""

    def write(self, topic: str, research_result: str, analysis_result: str) -> str:
        """調査・分析結果を統合して最終回答を執筆する"""
        return llm_chat([
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user', 'content':
                f'テーマ: {topic}\n\n'
                f'【調査結果】\n{research_result}\n\n'
                f'【分析結果】\n{analysis_result}\n\n'
                f'上記をもとに、テーマについての最終回答をまとめてください。'},
        ], max_tokens=512)


# エージェントのインスタンス化
research_agent = ResearchAgent()
analysis_agent = AnalysisAgent()
writing_agent  = WritingAgent()

print('✅ 3エージェント定義完了')
print(f'  {research_agent.name}')
print(f'  {analysis_agent.name}')
print(f'  {writing_agent.name}')

## Step 6: マルチエージェント実行関数

In [ ]:
def run_multi_agent(topic: str, max_discussion_rounds: int = 2) -> dict:
    """
    3エージェントがディスカッションしながらタスクを処理する

    フロー:
      1. 調査エージェントが初回調査
      2. 分析エージェントが分析 → 追加調査依頼があれば調査エージェントに差し戻し
      3. 追加調査があれば調査エージェントが再調査（最大 max_discussion_rounds 回）
      4. 執筆エージェントが最終回答を作成
    """
    SEP  = '─' * 55
    SEP2 = '┄' * 55
    log  = []  # ディスカッションの記録

    print(f'\n{SEP}')
    print(f'🤝 マルチエージェント起動')
    print(f'📋 テーマ: {topic}')
    print(SEP)

    # ──────────────────────────────────
    # Phase 1: 調査エージェント（初回）
    # ──────────────────────────────────
    print(f'\n{research_agent.name}')
    print('  → 初回調査中...')
    research_result = research_agent.research(topic)
    print(f'  ✓ {research_result[:150]}...' if len(research_result) > 150 else f'  ✓ {research_result}')
    log.append({'agent': research_agent.name, 'round': 0, 'output': research_result})

    # ──────────────────────────────────────────────────────
    # Phase 2: 分析エージェント ⇄ 調査エージェント（ディスカッション）
    # ──────────────────────────────────────────────────────
    analysis_result = None
    for round_num in range(1, max_discussion_rounds + 1):
        print(f'\n{SEP2}')
        print(f'{analysis_agent.name}  [ラウンド {round_num}]')
        print('  → 分析中...')
        result = analysis_agent.analyze(topic, research_result)
        analysis_result = result['analysis']
        additional_request = result['additional_request']

        print(f'  ✓ {analysis_result[:150]}...' if len(analysis_result) > 150 else f'  ✓ {analysis_result}')
        log.append({'agent': analysis_agent.name, 'round': round_num, 'output': analysis_result})

        if additional_request:
            # 追加調査依頼 → 調査エージェントに差し戻し
            print(f'\n  💬 追加調査依頼: {additional_request[:100]}')
            print(f'\n{research_agent.name}  [追加調査 ラウンド {round_num}]')
            print('  → 追加調査中...')
            research_result = research_agent.research(topic, additional_request)
            print(f'  ✓ {research_result[:150]}...' if len(research_result) > 150 else f'  ✓ {research_result}')
            log.append({'agent': research_agent.name, 'round': round_num, 'output': research_result})
        else:
            # 追加調査不要 → ディスカッション終了
            print('  ✅ 追加調査不要 → ディスカッション終了')
            break

    # ──────────────────────────────────
    # Phase 3: 執筆エージェント
    # ──────────────────────────────────
    print(f'\n{SEP2}')
    print(f'{writing_agent.name}')
    print('  → 最終回答を執筆中...')
    final_answer = writing_agent.write(topic, research_result, analysis_result)
    log.append({'agent': writing_agent.name, 'round': 'final', 'output': final_answer})

    print(f'\n{SEP}')
    print('✅ 最終回答')
    print(SEP)
    print(final_answer)
    print(SEP)

    return {'topic': topic, 'log': log, 'final_answer': final_answer}


print('✅ マルチエージェント実行関数 定義完了')

## Step 7: 実行してみよう！

In [ ]:
# 例1: 機械学習について
result = run_multi_agent("機械学習")

In [ ]:
# 例2: Phi-3.5とllama-cpp-pythonについて
result = run_multi_agent("Phi-3.5とllama-cpp-python")

In [ ]:
# 例3: 自由入力
MY_TOPIC = "深層学習"  # ← ここを変えて試してください
result = run_multi_agent(MY_TOPIC)

## Step 8: ディスカッションログの確認

In [ ]:
# 各エージェントの発言を時系列で確認
print('\n📜 ディスカッションログ')
print('=' * 55)
for entry in result['log']:
    print(f"\n[{entry['agent']}] (ラウンド: {entry['round']})")
    print(entry['output'])
    print('┄' * 55)

## Step 9: インタラクティブUI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

topic_box = widgets.Text(
    value='機械学習',
    description='テーマ:',
    layout=widgets.Layout(width='60%'),
)
rounds_slider = widgets.IntSlider(
    value=2, min=1, max=3, step=1,
    description='最大ラウンド数:',
    layout=widgets.Layout(width='60%'),
)
run_btn = widgets.Button(
    description='🤝 マルチエージェント実行',
    button_style='primary',
    layout=widgets.Layout(width='240px', height='38px'),
)
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        run_multi_agent(topic_box.value, max_discussion_rounds=rounds_slider.value)

run_btn.on_click(on_click)
display(topic_box, rounds_slider, run_btn, out)

---
## 📚 カスタマイズガイド

### エージェントの役割を変える
各エージェントの `system_prompt` を書き換えるだけで役割を変更できます。
```python
research_agent.system_prompt = "あなたは法律専門のリサーチャーです..."
analysis_agent.system_prompt = "あなたはリスク分析の専門家です..."
writing_agent.system_prompt  = "あなたは法律文書の執筆専門家です..."
```

### エージェントを追加する
```python
class ReviewAgent:
    def __init__(self):
        self.name = '🔎 レビューエージェント'
        self.system_prompt = '最終回答の誤りや漏れを指摘してください。'

    def review(self, topic, final_answer):
        return llm_chat([...], max_tokens=300)

review_agent = ReviewAgent()
```

### ディスカッションのラウンド数を増やす
```python
run_multi_agent("テーマ", max_discussion_rounds=3)
```